# Pre-Procesamiento de Datos

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from utils import get_data

Una vista al dataset:

In [ ]:
# Leemos el dataset
df = get_data()

df.head()

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,10365064,HZ100370,2015-12-31 23:59:00,075XX S EMERALD AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,...,17.0,68.0,14,1172605.0,1854931.0,2015,2018-02-10 15:50:01,41.757367,-87.642993,POINT (-87.642992854 41.757366519)
1,10460641,HZ199559,2015-12-31 23:59:00,015XX N KEDZIE AVE,0890,THEFT,FROM BUILDING,RESIDENCE PORCH/HALLWAY,False,False,...,26.0,23.0,06,NaN,NaN,2015,2018-02-09 15:44:29,NaN,NaN,NaN
2,10364662,HZ100006,2015-12-31 23:55:00,079XX S STONY ISLAND AVE,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,STREET,False,False,...,8.0,45.0,04B,1188223.0,1852840.0,2015,2018-02-10 15:50:01,41.751270,-87.585822,POINT (-87.585822373 41.751270452)
3,10364740,HZ100010,2015-12-31 23:50:00,024XX W FARGO AVE,0820,THEFT,$500 AND UNDER,APARTMENT,False,False,...,50.0,2.0,06,1158878.0,1949369.0,2015,2018-02-10 15:50:01,42.016804,-87.690709,POINT (-87.690708662 42.016804165)
4,10364683,HZ100002,2015-12-31 23:50:00,037XX N CLARK ST,0460,BATTERY,SIMPLE,SIDEWALK,True,False,...,44.0,6.0,08B,1167786.0,1925033.0,2015,2018-02-10 15:50:01,41.949837,-87.658635,POINT (-87.658635101 41.949837364)


In [ ]:
# Eliminamos las variables redundantes para dejar solo los features
df = df.drop([
    # Identificadores
    'ID',

    # Fue asignado posterior al hecho
    'Updated On',

    # Tiene el mismo valor para todas las filas
    'Year',

    # Redundantes con respecto a X Coordinate y Y Coordinate
    'Latitude',
    'Longitude',
    'Location',
], axis=1)

# Eliminamos date y cambiamos por month, day, hour y weekday
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Hour'] = df['Date'].dt.hour
df['Weekday'] = df['Date'].dt.weekday
df = df.drop(['Date'], axis=1)

Vamos a investigar crímenes que aparecen duplicados en el dataset. 

In [ ]:
df_duplicates = df[df.duplicated(subset=['Case Number'], keep=False)]
df_unique = df.drop_duplicates(subset=['Case Number'], keep='first')

display(df_duplicates.head())
display(df_duplicates['Primary Type'].value_counts())
rows = df.shape[0]

,Case Number,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Month,Day,Hour,Weekday
2840,HY551250,059XX S WENTWORTH AVE,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,232,2,20.0,68.0,01A,1175978.0,1865746.0,12,27,6,6
2910,HY551250,059XX S WENTWORTH AVE,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,232,2,20.0,68.0,01A,1175978.0,1865746.0,12,27,2,6
66152,HY442430,003XX W 42ND ST,0110,HOMICIDE,FIRST DEGREE MURDER,PARK PROPERTY,True,False,925,9,3.0,37.0,01A,1174451.0,1877100.0,9,29,3,1
66177,HY442430,003XX W 42ND ST,0110,HOMICIDE,FIRST DEGREE MURDER,PARK PROPERTY,True,False,925,9,3.0,37.0,01A,1174451.0,1877100.0,9,29,1,1
66188,HY442430,003XX W 42ND ST,0110,HOMICIDE,FIRST DEGREE MURDER,PARK PROPERTY,True,False,925,9,3.0,37.0,01A,1174451.0,1877100.0,9,29,1,1


Primary Type
HOMICIDE    50
Name: count, dtype: int64

Corresponden a homicidios múltiples (Ver [About Data](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data)). Vamos a verificar si vale la pena conservar esta información.

Podríamos crear algo así de forma directa:

```python
# Antes de dropear 'Case Number': conservamos la info de víctimas múltiples
case_size = df.groupby('Case Number')['Case Number'].transform('size')
df_unique['MULTIPLE_VICTIMS'] = (case_size > 1)
```

Pero hay que considerar a qué porcentaje del dataset corresponde esta información. Si es un porcentaje muy bajo, no vale la pena conservarla.

In [ ]:
display(df['Primary Type'].value_counts())
display(df.shape)

Primary Type
THEFT                                57355
BATTERY                              48927
CRIMINAL DAMAGE                      28677
NARCOTICS                            23939
OTHER OFFENSE                        17566
ASSAULT                              17048
DECEPTIVE PRACTICE                   16502
BURGLARY                             13184
MOTOR VEHICLE THEFT                  10068
ROBBERY                               9638
CRIMINAL TRESPASS                     6401
WEAPONS VIOLATION                     3364
OFFENSE INVOLVING CHILDREN            2527
PUBLIC PEACE VIOLATION                2422
PROSTITUTION                          1322
CRIM SEXUAL ASSAULT                   1319
INTERFERENCE WITH PUBLIC OFFICER      1308
SEX OFFENSE                           1054
HOMICIDE                               502
ARSON                                  453
GAMBLING                               310
LIQUOR LAW VIOLATION                   292
KIDNAPPING                             19

(264910, 19)

Vamos a crear de forma sintética un NUM_VICTIMS, que nos indique la cantidad de víctimas por caso. Esto nos permitirá analizar si es relevante o no conservar esta información.

In [ ]:
# Los Case Number repetidos son homicidios con múltiples víctimas:
# cada fila representa una víctima del mismo caso (27 casos, 50 filas — ver EDA).
# Trabajamos a nivel de caso, conservando una fila por Case Number.
df_unique = df.drop_duplicates(subset=['Case Number'], keep='first')

# Agregamos información de cantidad de víctimas por caso.
case_size = df.groupby('Case Number')['Case Number'].transform('size')
df_unique['NUM_VICTIMS'] = case_size
display(df_unique.head())

# Check df where NUM_VICTIMS > 1
display(df_unique[df_unique['NUM_VICTIMS'] > 1].head())

,Case Number,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Month,Day,Hour,Weekday,NUM_VICTIMS
0,HZ100370,075XX S EMERALD AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,False,621,6,17.0,68.0,14,1172605.0,1854931.0,12,31,23,3,1
1,HZ199559,015XX N KEDZIE AVE,0890,THEFT,FROM BUILDING,RESIDENCE PORCH/HALLWAY,False,False,1422,14,26.0,23.0,06,NaN,NaN,12,31,23,3,1
2,HZ100006,079XX S STONY ISLAND AVE,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,STREET,False,False,411,4,8.0,45.0,04B,1188223.0,1852840.0,12,31,23,3,1
3,HZ100010,024XX W FARGO AVE,0820,THEFT,$500 AND UNDER,APARTMENT,False,False,2411,24,50.0,2.0,06,1158878.0,1949369.0,12,31,23,3,1
4,HZ100002,037XX N CLARK ST,0460,BATTERY,SIMPLE,SIDEWALK,True,False,1923,19,44.0,6.0,08B,1167786.0,1925033.0,12,31,23,3,1


,Case Number,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Month,Day,Hour,Weekday,NUM_VICTIMS
2840,HY551250,059XX S WENTWORTH AVE,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,232,2,20.0,68.0,01A,1175978.0,1865746.0,12,27,6,6,2
66152,HY442430,003XX W 42ND ST,0110,HOMICIDE,FIRST DEGREE MURDER,PARK PROPERTY,True,False,925,9,3.0,37.0,01A,1174451.0,1877100.0,9,29,3,1,3
66379,HY442160,053XX S ABERDEEN ST,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,False,False,934,9,16.0,61.0,01A,1169890.0,1869399.0,9,28,20,0,2
86976,HY406845,050XX N WINTHROP AVE,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,True,False,2024,20,48.0,3.0,01A,1167957.0,1933812.0,9,2,2,2,2
90101,HY402004,052XX S LOREL AVE,0110,HOMICIDE,FIRST DEGREE MURDER,HOUSE,False,False,814,8,23.0,56.0,01A,1141580.0,1869447.0,8,29,9,5,2


Los homicidios múltiples corresponden a un 10% de los casos de homicidio.
Ahora estos corresponden a un 0.01% de los casos totales. Por lo tanto, consideramos que no vale la pena conservar esta información (**pero vale tenerlo en consideración**).